# 1 环境依赖

Use the environment of NIMBUS, run `conda activate Nimbus`

# 2 数据目录结构

在每个slides的独立的文件夹下建立一个`segmentation/Nimbus_output`的文件夹用来保存Nimbus的输出



# 3 处理逻辑

NIMBUS

# 4 原始代码

/home/jqyang/Nimbus-Inference/templates/1_Nimbus_Predict_IBD_Batch.ipynb

/home/jqyang/Nimbus-Inference/templates/1_Nimbus_Predict_IBD_PerSample.ipynb

In [1]:
import warnings
warnings.simplefilter("ignore")
import os
import pandas as pd
from IPython.display import display, HTML
from nimbus_inference.nimbus import Nimbus, prep_naming_convention
from nimbus_inference.utils import MultiplexDataset
from alpineer import io_utils

# --- 设置参数 (全局通用) ---
# 定义需要包含的通道
INCLUDE_CHANNELS = [
    "CD3", "CD4", "CD8a", "TCRbeta", "CXCR4", "EOMES"
]

# 定义数据所在的父目录 (假设 list.txt 里只有文件夹名，例如 "B514A-TI-N_ALN_rmbg-cellpose")
# 如果 list.txt 里是完整路径，请相应修改此处逻辑
DATA_ROOT = "../data" 

# list.txt 文件的路径
LIST_FILE = "list.txt"

# --- 读取 list.txt ---
try:
    with open(LIST_FILE, 'r') as f:
        # 读取每一行，去除空白符，并忽略空行
        folder_list = [line.strip() for line in f if line.strip()]
    print(f"Found {len(folder_list)} folders to process in {LIST_FILE}.")
except FileNotFoundError:
    print(f"Error: {LIST_FILE} not found. Please ensure the file exists.")
    folder_list = []

# --- 批量处理循环 ---
for i, folder_name in enumerate(folder_list):
    print(f"\n{'='*20}")
    print(f"Processing sample {i+1}/{len(folder_list)}: {folder_name}")
    print(f"{'='*20}")

    # 动态构建当前样本的 base_dir
    # 如果 list.txt 里已经是绝对路径或相对路径，可以直接用 base_dir = folder_name
    base_dir = os.path.normpath(os.path.join(DATA_ROOT, folder_name))

    # 检查目录是否存在
    if not os.path.exists(base_dir):
        print(f"Skipping: Directory not found -> {base_dir}")
        continue

    try:
        # 1. 设置文件路径
        tiff_dir = os.path.join(base_dir, "image_data")
        deepcell_output_dir = os.path.join(base_dir, "segmentation", "deepcell_output")
        nimbus_output_dir = os.path.join(base_dir, "nimbus_output")

        # 创建输出目录
        os.makedirs(nimbus_output_dir, exist_ok=True)

        # 验证路径是否存在 (io_utils 会抛出错误如果路径不存在)
        io_utils.validate_paths([base_dir, tiff_dir, deepcell_output_dir, nimbus_output_dir])

        # 2. 获取 FOV 列表
        fov_names = os.listdir(tiff_dir)
        # 过滤掉隐藏文件 (如 .DS_Store)
        fov_names = [fov for fov in fov_names if not fov.startswith(".")]
        fov_paths = [os.path.join(tiff_dir, fov) for fov in fov_names]
        
        print(f"Found {len(fov_names)} FOVs.")

        # 3. 准备 Segmentation Naming Convention
        # 这一步会为当前样本生成对应的命名规则函数
        segmentation_naming_convention = prep_naming_convention(deepcell_output_dir)

        # 简单的验证
        if len(fov_paths) > 0:
            test_seg_path = segmentation_naming_convention(fov_paths[0])
            if not os.path.exists(test_seg_path):
                print(f"Warning: Segmentation file not found for {fov_paths[0]} at {test_seg_path}")

        # 4. 初始化 Dataset
        dataset = MultiplexDataset(
            fov_paths=fov_paths,
            suffix=".tif", # 请根据实际情况确认是 .tif 还是 .ome.tiff
            include_channels=INCLUDE_CHANNELS,
            segmentation_naming_convention=segmentation_naming_convention,
            output_dir=nimbus_output_dir,
        )

        # 5. 初始化 Nimbus 模型
        # 注意：虽然在循环内重复加载模型可能会增加一点开销，
        # 但这样能确保 dataset 对象的正确绑定，避免上一个样本的状态污染。
        nimbus = Nimbus(
            dataset=dataset,
            save_predictions=True,
            batch_size=4,        # 如果显存不够，改小这个数字
            test_time_aug=True,  # 如果没有 GPU 或速度太慢，设为 False
            input_shape=[1024, 1024],
            device="auto",
            checkpoint="V1.pt",  # 确保 V1.pt 在当前目录或指定绝对路径
            output_dir=nimbus_output_dir,
        )

        # 检查输入
        nimbus.check_inputs()

        # 6. 准备归一化字典
        print("Preparing normalization dictionary...")
        dataset.prepare_normalization_dict(
            quantile=0.999,
            n_subset=50,
            clip_values=(0, 2),
            multiprocessing=True,
            overwrite=True
        )

        # 7. 执行预测
        print("Running prediction...")
        cell_table = nimbus.predict_fovs()
        
        # 打印部分结果以确认
        print(f"Prediction complete for {folder_name}. Table shape: {cell_table.shape}")
        
        # 显式保存一下 csv (虽然 Nimbus 内部可能已经保存了，但这样更保险)
        output_csv_path = os.path.join(nimbus_output_dir, "nimbus_cell_table.csv")
        print(f"Results saved to: {output_csv_path}")

    except Exception as e:
        print(f"!!! Error processing {folder_name}: {e}")
        import traceback
        traceback.print_exc()
        continue # 出错后跳过当前样本，继续下一个

print("\nBatch processing finished.")

Found 10 folders to process in list.txt.

Processing sample 1/10: A099A-CXCR4-cellpose
Found 71 FOVs.
All inputs are valid
Loaded weights from /home/jqyang/Nimbus-Inference/src/nimbus_inference/assets/V1.pt
All inputs are valid.
Preparing normalization dictionary...
Iterate over fovs...
Running prediction...
Available GPUs:  1
Predictions will be saved in ../data/A099A-CXCR4-cellpose/nimbus_output
Iterating through fovs will take a while...
Predicting ../data/A099A-CXCR4-cellpose/image_data/fov68...


  0%|          | 0/6 [00:00<?, ?it/s]

Predicting ../data/A099A-CXCR4-cellpose/image_data/fov69...


  0%|          | 0/6 [00:00<?, ?it/s]

Predicting ../data/A099A-CXCR4-cellpose/image_data/fov37...


...

Predicting ../data/B581A-CXCR4-cellpose/image_data/fov28...


  0%|          | 0/6 [00:00<?, ?it/s]

Predicting ../data/B581A-CXCR4-cellpose/image_data/fov32...


  0%|          | 0/6 [00:00<?, ?it/s]

Prediction complete for B581A-CXCR4-cellpose. Table shape: (27342, 8)
Results saved to: ../data/B581A-CXCR4-cellpose/nimbus_output/nimbus_cell_table.csv

Batch processing finished.
